<a href="https://colab.research.google.com/github/AntonRize/WILL/blob/main/Colab_Notebooks/RELATIONAL_ORBITAL_HOLOGRAPHY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. RUN !pip install emcee

In [ ]:
!pip install emcee

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.1 MB/s eta 0:00:00


2. INPUT RANDOM_SEED = xxxx  AND RUN 1PN DATA GENERATOR

In [ ]:
# ==========================================
#  1PN DATA GENERATOR
# ==========================================

import numpy as np
import pandas as pd

# ==========================================
# 0. REPRODUCIBILITY
# ==========================================
RANDOM_SEED = 58714
np.random.seed(RANDOM_SEED)

G = 6.67430e-11
C = 299792458.0
M_SUN = 1.98847e30
SEC_PER_YEAR = 365.25 * 86400

print(f"Initializing 1PN Generative Engine (Seed: {RANDOM_SEED})...")
print("Scanning parameter space for viable relational signal...\n")

while True:
    M_bh = np.random.uniform(3.8e6, 4.2e6) * M_SUN
    P_yrs = np.random.uniform(12.0, 16.0)
    e_true = np.random.uniform(0.88, 0.94)
    i_true = np.random.uniform(np.radians(120.0), np.radians(150.0))
    w_true = np.random.uniform(0.0, 2 * np.pi)
    vz0_true = np.random.uniform(-10.0, 10.0)
    T_peri = 58000.0
    NOISE_SIGMA_KMS = 3.0

    P_sec = P_yrs * SEC_PER_YEAR
    a_meters = (G * M_bh * (P_sec**2) / (4 * np.pi**2))**(1/3)

    # 1. Observation Epochs (MUST include TWO periapsis passages to capture precession)
    t_bg = np.random.uniform(T_peri - (P_yrs * 365 * 0.5), T_peri + (P_yrs * 365 * 1.5), 40)
    t_peri_1 = np.random.normal(T_peri, P_yrs * 365 * 0.05, 60)
    t_peri_2 = np.random.normal(T_peri + P_yrs * 365.25, P_yrs * 365 * 0.05, 60)
    t_obs = np.sort(np.concatenate([t_bg, t_peri_1, t_peri_2]))

    # 2. Stable Keplerian Engine with Unwrapped True Anomaly
    M_unwrapped = (2 * np.pi / P_sec) * (t_obs * 86400 - T_peri * 86400)
    orbit_count = np.floor(M_unwrapped / (2 * np.pi))
    M_wrapped = M_unwrapped % (2 * np.pi)
    E = M_wrapped.copy()

    for _ in range(50):
        f = E - e_true * np.sin(E) - M_wrapped
        dE = f / (1 - e_true * np.cos(E))
        E -= dE
        if np.max(np.abs(dE)) < 1e-12:
            break

    nu_wrapped = 2 * np.arctan2(np.sqrt(1 + e_true) * np.sin(E / 2), np.sqrt(1 - e_true) * np.cos(E / 2))
    nu_positive = nu_wrapped % (2 * np.pi)
    nu_unwrapped = nu_positive + orbit_count * 2 * np.pi

    r_obs = a_meters * (1 - e_true**2) / (1 + e_true * np.cos(nu_positive))

    # 3. 1PN Additive Physics (CONTINUOUS ACCUMULATION per radian)
    delta_w_per_radian = (3 * G * M_bh) / (a_meters * (1 - e_true**2) * C**2)
    w_dynamic = w_true + delta_w_per_radian * nu_unwrapped

    K_amp = np.sqrt(G * M_bh / (a_meters * (1 - e_true**2))) * np.sin(i_true)
    v_rad = K_amp * (np.cos(nu_positive + w_dynamic) + e_true * np.cos(w_dynamic))

    v_orb_sq_t = G * M_bh * (2.0 / r_obs - 1.0 / a_meters)
    z_grav = (G * M_bh) / (r_obs * C**2)
    z_transverse = v_orb_sq_t / (2 * C**2)
    v_rel_shift_kms = (z_grav + z_transverse) * C / 1000.0

    # 4. Strict Signal Validation
    rel_shift_amplitude = np.max(v_rel_shift_kms) - np.min(v_rel_shift_kms)
    snr_rel = rel_shift_amplitude / NOISE_SIGMA_KMS

    if snr_rel >= 4.0:
        break

# 5. Finalizing Observables
rv_clean_kms = vz0_true + (v_rad / 1000.0) + v_rel_shift_kms
rv_noisy_kms = rv_clean_kms + np.random.normal(0, NOISE_SIGMA_KMS, len(t_obs))
sigma_array = np.full(len(t_obs), NOISE_SIGMA_KMS)

# 6. Quality Report
print("=================================================================")
print("DATASET VALIDATION REPORT")
print("=================================================================")
print(f"Total Epochs Sampled:      {len(t_obs)}")
print(f"Observation Baseline:      {(np.max(t_obs) - np.min(t_obs))/365.25:.2f} years")
print(f"Noise Floor (1 sigma):     {NOISE_SIGMA_KMS:.1f} km/s")
print("-----------------------------------------------------------------")
print(f"Classical Semi-Amplitude:  {np.abs(K_amp)/1000.0:.1f} km/s")
print(f"Relativistic Amplitude:    {rel_shift_amplitude:.2f} km/s")
print(f"Relativistic SNR:          {snr_rel:.2f}")
print("=================================================================")
print("STATUS: SUCCESS. Data meets strict criteria for blind testing.\n")

# 7. Data Export
pd.DataFrame({
    'MJD': t_obs,
    'RV_km_s': rv_noisy_kms,
    'sigma_km_s': sigma_array
}).to_csv('GR_BLIND_DATASET.csv', index=False)

pd.DataFrame([{
    'P_yrs': P_yrs,
    'e_true': e_true,
    'w_true_deg': np.degrees(w_true) % 360,
    'i_true_deg': np.degrees(i_true) % 180,
    'vz0_true': vz0_true
}]).to_csv('GR_BLIND_TRUTH.csv', index=False)

print("Exported 'GR_BLIND_DATASET.csv' (Observables only).")
print("Exported 'GR_BLIND_TRUTH.csv' (Hidden Truth).")

Initializing 1PN Generative Engine (Seed: 1274)...
Scanning parameter space for viable relational signal...

DATASET VALIDATION REPORT
Total Epochs Sampled:      160
Observation Baseline:      22.76 years
Noise Floor (1 sigma):     3.0 km/s
-----------------------------------------------------------------
Classical Semi-Amplitude:  3195.5 km/s
Relativistic Amplitude:    437.88 km/s
Relativistic SNR:          145.96
STATUS: SUCCESS. Data meets strict criteria for blind testing.

Exported 'GR_BLIND_DATASET.csv' (Observables only).
Exported 'GR_BLIND_TRUTH.csv' (Hidden Truth).


**3. RUN R.O.M. HOLGRAPHIC DECODER TO RECOVER:**

Period (P)

Eccentricity (e)

Inclination   (i)   

Background Drift (km/s)

R.O.M. Kinematic Proj at semi-major axis (beta)

In [ ]:
# ==========================================
#  R.O.M. HOLGRAPHIC DECODER
# ==========================================

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
import emcee
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------
C_KMS = 299792.458 # Speed of light in km/s

# ---------------------------------------------------------
# 0. LOAD DATA
# ---------------------------------------------------------
print("Loading GR_BLIND_DATASET.csv...")
df = pd.read_csv('GR_BLIND_DATASET.csv')
t_obs = df['MJD'].values
vz_obs = df['RV_km_s'].values
sigma_vz = df['sigma_km_s'].values

Z_obs = 1.0 + (vz_obs / C_KMS)
sigma_Z = sigma_vz / C_KMS

idx_peak = np.argmax(np.abs(vz_obs))
t_peak = t_obs[idx_peak]

# ---------------------------------------------------------
# 1. R.O.M. EXACT GEOMETRY ENGINE (CONTINUOUS)
# ---------------------------------------------------------
def get_phase_unwrapped(t, t_peri, P, e):
    M_unwrapped = (2 * np.pi / P) * (t - t_peri)
    orbit_count = np.floor(M_unwrapped / (2 * np.pi))
    M_wrapped = M_unwrapped % (2 * np.pi)

    E = M_wrapped.copy()
    for _ in range(30):
        f = E - e * np.sin(E) - M_wrapped
        dE = f / (1 - e * np.cos(E))
        E -= dE
        if np.max(np.abs(dE)) < 1e-10:
            break

    o_wrapped = 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))
    o_positive = o_wrapped % (2 * np.pi)
    o_unwrapped = o_positive + orbit_count * 2 * np.pi
    return o_unwrapped

def generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i_inc, beta_z0, e, omega_0, P_days, T_peri):
    # EXACT WILL RELATIONAL GEOMETRY ACCUMULATION (No truncation)
    # tau^2 = (1 - beta^2)(1 - kappa^2) where kappa^2 = 2*beta^2
    tau_sq = (1.0 - beta**2) * (1.0 - 2.0 * beta**2)

    # tau_Y^2 = 1 - tau^2 (Strict divergence including cross-coupling term)
    tau_Y_sq = 1.0 - tau_sq

    # Exact geometric phase shift per radian, normalized by shape factor
    omega_dynamic = omega_0 + (tau_Y_sq / (1.0 - e**2)) * o_unwrapped

    o = o_unwrapped % (2 * np.pi)

    # LOS Doppler Amplitude Calculation
    beta_int = beta / np.sqrt(1 - e**2)
    K = beta_int * np.sin(i_inc)
    beta_los = K * (np.cos(o + omega_dynamic) + e * np.cos(omega_dynamic))

    # Second-order systemic relational invariants (Independent of inclination)
    beta_o_sq = (beta**2) * (1 + e**2 + 2 * e * np.cos(o)) / (1 - e**2)
    kappa_o_sq = 2 * (beta**2) * (1 + e * np.cos(o)) / (1 - e**2)

    Z_sys = (1 - beta_o_sq)**(-0.5) * (1 - kappa_o_sq)**(-0.5)

    return Z_sys * (1 + beta_los) * (1 + beta_z0)

# ---------------------------------------------------------
# 2. STAGE 1: KEPLERIAN SCOUT
# ---------------------------------------------------------
def classical_objective(params):
    K_guess, vz0_guess, e_guess, omega_guess, P_days_guess, t_peri_guess = params
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri_guess, P_days_guess, e_guess)
        o_obs = o_unwrapped % (2 * np.pi)
        v_rad_kms = K_guess * (np.cos(o_obs + omega_guess) + e_guess * np.cos(omega_guess)) + vz0_guess
        Z_model_class = 1.0 + (v_rad_kms / C_KMS)
        return np.sum(((Z_obs - Z_model_class) / sigma_Z)**2)
    except:
        return 1e10

bounds_stage1 = [
    (100.0, 15000.0), (-50.0, 50.0), (0.80, 0.99),
    (0.0, 2 * np.pi), (3000.0, 7000.0), (t_peak - 100.0, t_peak + 100.0)
]

print(f"STAGE 1: Keplerian Scout (Locating P, e, T_peri)...")
res_s1 = differential_evolution(classical_objective, bounds_stage1, strategy='best1bin', maxiter=500, popsize=20, tol=1e-4, seed=101)
K_s1, vz0_s1, e_s1, omega_s1, P_s1, t_peri_s1 = res_s1.x
print(f"-> Skeleton Found: P = {P_s1/365.25:.2f} yrs, e = {e_s1:.4f}")

# ---------------------------------------------------------
# 3. STAGE 2: R.O.M. GLOBAL SNIPER
# ---------------------------------------------------------
def rom_objective(params):
    beta, i, vz0_c, e, omega, P, t_peri = params
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri, P, e)
        Z_model = generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i, vz0_c, e, omega, P, t_peri)
        return np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return 1e10

# Bounding inclination strictly to [0, pi/2] breaks the mirror degeneracy.
bounds_stage2 = [
    (0.001, 0.05),                                    # beta
    (0.001, np.pi/2),                                 # i (Folded domain)
    (-50.0/C_KMS, 50.0/C_KMS),                        # vz0_c
    (max(0.8, e_s1 - 0.05), min(0.999, e_s1 + 0.05)), # e (Constrained by S1)
    (0.0, 2 * np.pi),                                 # omega
    (max(3000.0, P_s1 - 200.0), P_s1 + 200.0),        # P (Constrained by S1)
    (t_peri_s1 - 50.0, t_peri_s1 + 50.0)              # T_peri (Constrained by S1)
]

print("\nSTAGE 2: Full R.O.M. Global Sniper (Deterministic Decoupling)...")
res_s2 = differential_evolution(rom_objective, bounds_stage2, strategy='best1bin', maxiter=1000, popsize=20, tol=1e-5, seed=101)
beta_s2, i_s2, vz0_c_s2, e_s2, omega_s2, P_s2, t_peri_s2 = res_s2.x
print(f"-> Exact Peak Found: beta = {beta_s2:.6f}, i = {np.degrees(i_s2):.2f} deg, vz0 = {vz0_c_s2*C_KMS:.2f} km/s")

# ---------------------------------------------------------
# 4. STAGE 3: MCMC MAPPING
# ---------------------------------------------------------
def log_prior(theta):
    beta, i, vz0_c, e, omega, P, t_peri = theta
    if not (0.0001 < beta < 0.1): return -np.inf
    if not (0.0 < i < np.pi/2): return -np.inf
    if not (0.8 < e < 0.999): return -np.inf
    if not (0.0 < omega < 2*np.pi): return -np.inf
    if not (3000.0 < P < 7000.0): return -np.inf
    if abs(vz0_c * C_KMS) > 50.0: return -np.inf
    return 0.0

def log_likelihood(theta):
    beta, i, vz0_c, e, omega, P, t_peri = theta
    try:
        o_unwrapped = get_phase_unwrapped(t_obs, t_peri, P, e)
        Z_model = generate_z_raw_dynamic(t_obs, o_unwrapped, beta, i, vz0_c, e, omega, P, t_peri)
        return -0.5 * np.sum(((Z_obs - Z_model) / sigma_Z)**2)
    except:
        return -np.inf

def log_probability(theta):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta)

print("\nSTAGE 3: MCMC Bayesian Mapping (Running 1000 steps)...")
nwalkers = 32
ndim = 7

# Exact initialization centered on the verified Global Sniper peak.
# Added microscopic Gaussian jitter to ensure matrix full rank.
initial_pos = np.array([beta_s2, i_s2, vz0_c_s2, e_s2, omega_s2, P_s2, t_peri_s2])
pos = initial_pos + 1e-6 * np.random.randn(nwalkers, ndim)

sampler = emcee.EnsembleSampler(nwalkers, ndim, log_probability)
sampler.run_mcmc(pos, 1000, progress=True)

flat_samples = sampler.get_chain(discard=300, thin=5, flat=True)
med_beta, med_i, med_vz0, med_e, med_omega, med_P, med_tperi = np.median(flat_samples, axis=0)

# ---------------------------------------------------------
# 5. VALIDATION OUTPUT
# ---------------------------------------------------------
df_truth = pd.read_csv('GR_BLIND_TRUTH.csv').iloc[0]

# Fold the true inclination into the [0, 90] domain for direct mathematical comparison
true_i_rad = np.radians(df_truth['i_true_deg'])
true_i_folded = true_i_rad if true_i_rad <= np.pi/2 else np.pi - true_i_rad

print(f"\n{'='*65}\nFINAL BLIND TEST RESULTS: EXACT R.O.M. GEOMETRY\n{'-'*65}")
print(f"{'Parameter':<25} | {'True (Hidden)':<15} | {'R.O.M. Median':<15}")
print(f"{'Period (yrs)':<25} | {df_truth['P_yrs']:<15.3f} | {med_P/365.25:<15.3f}")
print(f"{'Eccentricity (e)':<25} | {df_truth['e_true']:<15.5f} | {med_e:<15.5f}")
print(f"{'Inclination (folded) [deg]':<25} | {np.degrees(true_i_folded):<15.2f} | {np.degrees(med_i):<15.2f}")
print(f"{'Background Drift (km/s)':<25} | {df_truth['vz0_true']:<15.2f} | {med_vz0*C_KMS:<15.2f}")
print(f"{'-'*65}")
print(f"R.O.M. Kinematic Proj (beta): {med_beta:.6f}")
print(f"{'='*65}")

Loading GR_BLIND_DATASET.csv...
STAGE 1: Keplerian Scout (Locating P, e, T_peri)...
-> Skeleton Found: P = 12.01 yrs, e = 0.9364

STAGE 2: Full R.O.M. Global Sniper (Deterministic Decoupling)...
-> Exact Peak Found: beta = 0.006842, i = 33.13 deg, vz0 = 10.36 km/s

STAGE 3: MCMC Bayesian Mapping (Running 1000 steps)...


100%|██████████| 1000/1000 [00:11<00:00, 90.42it/s]


FINAL BLIND TEST RESULTS: EXACT R.O.M. GEOMETRY
-----------------------------------------------------------------
Parameter                 | True (Hidden)   | R.O.M. Median  
Period (yrs)              | 12.006          | 12.006         
Eccentricity (e)          | 0.93636         | 0.93625        
Inclination (folded) [deg] | 32.57           | 33.43          
Background Drift (km/s)   | 9.55            | 10.73          
-----------------------------------------------------------------
R.O.M. Kinematic Proj (beta): 0.006786


4. **RUN R.O.M. HOLOGRAPHIC EXTRACTOR** AND RECONSTRUCT **REALITY**

In [ ]:
import numpy as np

# ==========================================
# R.O.M. HOLOGRAPHIC EXTRACTOR
# Reconstructing absolute 3D reality from 1D relational data
# ==========================================

# FUNDAMENTAL CONSTANT (Only speed of light is needed for metric translation)
C = 299792458.0  # m/s

def reconstruct_system(P_years, e, beta, i_deg, omega_deg, v_z0_kms):
    print("="*60)
    print(" 1. INPUT: 1D EXTRACTED RELATIONAL INVARIANTS")
    print("="*60)
    print(f" Period (P):            {P_years:.4f} years")
    print(f" Eccentricity (e):      {e:.5f}")
    print(f" Kinematic Proj (beta): {beta:.6f}")
    print(f" Inclination (i):       {i_deg:.2f} deg")
    print(f" Background Drift:      {v_z0_kms:.2f} km/s")

    # ---------------------------------------------------------
    # STAGE 1: PURE DIMENSIONLESS GEOMETRY (No metric, no mass)
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print(" 2. R.O.M. STRUCTURAL SYMMETRY (Dimensionless)")
    print("="*60)

    # Global System Energy and Potential
    W = (beta**2) / 2.0
    kappa_sq = 2.0 * (beta**2)  # Structural closure theorem

    # Perihelion local projections
    kappa_p_sq = kappa_sq / (1 - e)
    beta_p = beta * np.sqrt((1 + e) / (1 - e))

    # Exact Precession (Accumulation of relational divergence)
    tau_sq = (1 - beta**2) * (1 - kappa_sq)
    tau_Y_sq = 1.0 - tau_sq
    precession_rad = (2 * np.pi * tau_Y_sq) / (1 - e**2)
    precession_deg = np.degrees(precession_rad)

    print(f" System Binding Energy (W):     {W:.4e}")
    print(f" Global Potential (kappa^2):    {kappa_sq:.4e}")
    print(f" Perihelion Potential (k_p^2):  {kappa_p_sq:.4e}")
    print(f" Perihelion Kinematic (beta_p): {beta_p:.6f}")
    print(f" Exact Orbital Precession:      {precession_deg:.4f} deg/orbit")

    # ---------------------------------------------------------
    # STAGE 2: ABSOLUTE METRIC TRANSLATION
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print(" 3. ABSOLUTE METRIC RECOVERY (Meters & Seconds)")
    print("="*60)

    P_sec = P_years * 365.25 * 86400.0

    # The Core R.O.M. Metric Extractor
    Rs_meters = (P_sec * C * (beta**3)) / np.pi

    # Macroscopic scale recovery
    a_meters = Rs_meters / kappa_sq
    r_p_meters = Rs_meters / kappa_p_sq

    # Velocity recovery
    v_p_kms = (beta_p * C) / 1000.0

    print(f" Schwarzschild Radius (Rs): {Rs_meters:,.2f} meters")
    print(f" Semi-major Axis (a):       {a_meters / 1e9:,.4f} million km")
    print(f" Perihelion Distance (r_p): {r_p_meters / 1e9:,.4f} million km")
    print(f" Peak Physical Velocity:    {v_p_kms:,.2f} km/s")

    # ---------------------------------------------------------
    # STAGE 3: LEGACY UNIT CONVERSION (Proving Redundancy)
    # ---------------------------------------------------------
    print("\n" + "="*60)
    print(" 4. LEGACY UNIT CONVERSION (Mass & Gravity)")
    print("="*60)

    G = 6.6743e-11
    M_sun = 1.98847e30

    # Recovering Mass from Geometry
    Mass_kg = (Rs_meters * C**2) / (2 * G)
    Mass_solar = Mass_kg / M_sun

    print(f" Derived System Mass:       {Mass_kg:.4e} kg")
    print(f" Mass in Solar Masses:      {Mass_solar:,.2f} M_sun")
    print("-" * 60)
    print(" NOTICE: G and kg were strictly absent from the generative")
    print(" algebraic chain. They are epistemologically and operationally redundant")
    print("="*60)

# ==========================================
# TEST RUN: Using the 4-orbit blind test data
# ==========================================
if __name__ == "__main__":
    # Inputting the extracted values from our previous successful blind MCMC
    reconstruct_system(
        P_years=15.200,
        e=0.84000,
        beta=0.006661,
        i_deg=81.37,
        omega_deg=105.03,
        v_z0_kms=-21.43
    )